In [ ]:

# ============================================================
# PYTORCH COMPLETE REVISION — SINGLE CELL
# Topics: Tensor, Autograd, Layers, Model, Loss, Optimizer,
# Dataset, DataLoader, Training Loop, Predictions, Save & Load
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)

# ============================================================
# 1. TENSOR BASICS
# ============================================================

print("\n========== 1. TENSOR BASICS ==========")

x = torch.randn(3, 4)  # 3 rows, 4 columns; normal distribution

print("Tensor:\n", x)
print("Shape:", x.shape)
print("Dimensions:", x.ndim)
print("Data type:", x.dtype)

x = x.reshape(2, 6)
print("Reshaped tensor shape:", x.shape)


# ============================================================
# 2. AUTOGRAD: GRADIENT CALCULATION
# ============================================================

print("\n========== 2. AUTOGRAD ==========")

x_grad = torch.tensor(3.0, requires_grad=True)

y_grad = x_grad**2 + 5*x_grad

print("x:", x_grad)
print("y:", y_grad)

y_grad.backward()

print("Gradient dy/dx:", x_grad.grad)
# y = x^2 + 5x
# dy/dx = 2x + 5
# At x = 3, gradient = 11


# ============================================================
# 3. SINGLE LINEAR LAYER
# ============================================================

print("\n========== 3. LINEAR LAYER ==========")

layer = nn.Linear(3, 2) #isme 3 input feature aur 2 output feature or neuron

print("Layer:", layer)
print("Weight shape:", layer.weight.shape)
print("Weights:\n", layer.weight)
print("Bias shape:", layer.bias.shape)
print("Bias:\n", layer.bias)


# ============================================================
# 4. BUILD A NEURAL NETWORK
# ============================================================

print("\n========== 4. NEURAL NETWORK ==========")

model = nn.Sequential(
    nn.Linear(10, 5),
    nn.ReLU(),
    nn.Linear(5, 2)
)

x = torch.randn(4, 10)  # 4 samples, 10 features each

output = model(x)

print("Input shape:", x.shape)
print("Output:\n", output)
print("Output shape:", output.shape)  # [4, 2]


# ============================================================
# 5. LOSS FUNCTION
# ============================================================

print("\n========== 5. LOSS FUNCTION ==========")

loss_fn = nn.CrossEntropyLoss()

y_true = torch.tensor([0, 1, 1, 0])

loss = loss_fn(output, y_true)

print("Loss:", loss)
print("Loss value:", loss.item())


# ============================================================
# 6. OPTIMIZER + BACKPROPAGATION
# ============================================================

print("\n========== 6. OPTIMIZER ==========")

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# Clear old gradients
optimizer.zero_grad()

# Calculate gradients
loss.backward()

# Update weights and biases
optimizer.step()

# Get fresh predictions after parameter updates
new_output = model(x)

print("Output after update:\n", new_output)

# Correct way to inspect weights and biases
for name, param in model.named_parameters():
    print(name, param.shape)
    print(param)


# ============================================================
# 7. DATASET + DATALOADER
# ============================================================

print("\n========== 7. DATASET & DATALOADER ==========")

x_data = torch.randn(100, 10)
y_data = torch.randint(0, 2, (100,))

dataset = TensorDataset(x_data, y_data)

loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True
)

print("Dataset X shape:", x_data.shape)
print("Dataset y shape:", y_data.shape)
print("Number of samples:", len(dataset))
print("Number of batches:", len(loader))

for batch_x, batch_y in loader:
    print("Batch X shape:", batch_x.shape)
    print("Batch y shape:", batch_y.shape)
    print("First sample:", batch_x[0])
    break  # Only inspect the first batch


# ============================================================
# 8. TRAINING LOOP — 5 EPOCHS
# ============================================================

print("\n========== 8. TRAINING LOOP ==========")

for epoch in range(5):

    model.train()

    total_loss = 0.0
    total_samples = 0
    correct = 0

    for batch_x, batch_y in loader:

        optimizer.zero_grad()

        output = model(batch_x)

        loss = loss_fn(output, batch_y)

        loss.backward()

        optimizer.step()

        # Calculate average loss for the entire epoch
        batch_size = batch_x.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        # Prediction for this batch
        predictions = output.argmax(dim=1)
        correct += (predictions == batch_y).sum().item()

    avg_loss = total_loss / total_samples
    accuracy = correct / total_samples

    print(
        f"Epoch {epoch + 1} | "
        f"Average Loss: {avg_loss:.4f} | "
        f"Approx. Accuracy: {accuracy:.2%}"
    )


# ============================================================
# 9. PREDICTIONS, LABELS AND RAW OUTPUT
# ============================================================

print("\n========== 9. PREDICTIONS ==========")

model.eval()

with torch.no_grad():
    output = model(x_data)

predictions = output.argmax(dim=1)

print("Full output shape:", output.shape)  # [100, 2]
print("First 5 raw output rows:\n", output[:5])
print("Actual labels:", y_data[:5])
print("Predictions:", predictions[:5])

# Convert logits into class probabilities
probabilities = torch.softmax(output, dim=1)

print("First 5 class probabilities:\n", probabilities[:5])


# ============================================================
# 10. UNDERSTAND ARGMAX DIMENSIONS
# ============================================================

print("\n========== 10. ARGMAX DIMENSIONS ==========")

example = torch.tensor([
    [2.1, 0.8],
    [0.4, 1.7],
    [1.3, 0.9]
])

print("Example tensor:\n", example)

print("dim=1:", example.argmax(dim=1))
# Compare columns inside each row:
# One predicted class per sample -> [0, 1, 0]

print("dim=0:", example.argmax(dim=0))
# Compare rows inside each column:
# Index of maximum value in each column -> [0, 1]


# ============================================================
# 11. SAVE MODEL PARAMETERS
# ============================================================

print("\n========== 11. SAVE MODEL ==========")

torch.save(model.state_dict(), "my_model.pth")

print("Model parameters saved to my_model.pth")


# ============================================================
# 12. LOAD MODEL PARAMETERS
# ============================================================

print("\n========== 12. LOAD MODEL ==========")

# Architecture must already exist and match the saved state
state = torch.load("my_model.pth", weights_only=True)

result = model.load_state_dict(state)

print(result)
print("Parameter keys:", model.state_dict().keys())


# ============================================================
# 13. INSPECT ALL PARAMETER SHAPES
# ============================================================

print("\n========== 13. PARAMETER SHAPES ==========")

for name, param in model.named_parameters():
    print(name, param.shape)


# ============================================================
# 14. VERIFY SAVE AND LOAD
# ============================================================

print("\n========== 14. VERIFY SAVE/LOAD ==========")

model.eval()

with torch.no_grad():
    before = model(x_data)

torch.save(model.state_dict(), "my_model.pth")

model.load_state_dict(
    torch.load("my_model.pth", weights_only=True)
)

model.eval()

with torch.no_grad():
    after = model(x_data)

print("Predictions unchanged:", torch.allclose(before, after))

print("\n========== REVISION COMPLETE ==========")
